ベース：白地図 CARTO Light No Labels

国境線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

人口ラスタ：WorldPop 'Syrian Arab Republic 100m Population 2026'
https://hub.worldpop.org/geodata/summary?id=75632
syr_worldpop_2026.tif

分析手法：Windowed Reading

対象地域：
・Syria (National Scale)

その他：
・Rasterio Windowクラスを使用し、GeoTIFF全体を読み込まず対象領域のみを抽出
・NoData値をnp.nanへ変換
・Window Boundsを取得しFoliumへ投影
・解像度を維持したままメモリ使用量を削減

In [1]:
import rasterio
from rasterio.windows import Window

import folium

import geopandas as gpd      

import numpy as np

import matplotlib.colors as mcolors

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ETL Extract
pop_data = 'tif_1_syria_worldpop_2026.tif'

with rasterio.open(pop_data) as src:
    my_window = Window(0, 0, 3000, 3000)

    # ETL Extract
    # # Windowed Reading
    # 指定領域のみを抽出して読み込む
    data_window = src.read(1, window=my_window).astype("float32")

    # ETL Transform
    # NoData値を欠損値として処理
    nodata = src.nodata
    data_window[data_window == nodata] = np.nan

    # ETL Transform
    # Window領域の座標情報を取得
    transform_window = src.window_transform(my_window)
    lon_left, lat_bottom, lon_right, lat_top = src.window_bounds(my_window)

    print("NoData value:", nodata)
    print("min:", np.nanmin(data_window))
    print("max:", np.nanmax(data_window))
    print("bounds:", lon_left, lat_bottom, lon_right, lat_top)

# ETL Transform
# Folium描画用にNaNを0へ変換
data_window_clean = np.nan_to_num(data_window, nan=0)

NoData value: -99999.0
min: 0.0
max: 463.28693
bounds: 35.7166658038 34.820000196719995 38.2166657938 37.320000186719994


In [3]:
# QGIS　オリジナルカラーコード

colors = [
    (0.00, "#7bb5a000"), 
    (0.20, "#7bb5a0ff"), 
    (0.40, "#2e9166ff"), 
    (0.60, "#226c4cff"), 
    (0.80, "#184d36ff"), 
    (1.00, "#0f3122ff")  
]
marisa_cmap = mcolors.LinearSegmentedColormap.from_list("Syria_Neon", colors)

In [4]:
# 白地図ベース

m = folium.Map(
    location=[34.8, 38.5], 
    zoom_start=7,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
)

In [5]:
# 国境線：太さ3
folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'black', 'weight': 3, 'fillOpacity': 0}).add_to(m)

# 地域線：太さ1
folium.GeoJson('syr_admin1.geojson', name='Governorate',
               style_function=lambda x: {'color': 'gray', 'weight': 1, 'fillOpacity': 0}).add_to(m)

In [6]:
# Raster Bounds
bounds_window = [
    [lat_bottom, lon_left],
    [lat_top, lon_right]
]

folium.raster_layers.ImageOverlay(
    image=data_window_clean,
    bounds=bounds_window,
    colormap=marisa_cmap,
    opacity=0.8,
    name='Population Density (Windowed)'
).add_to(m)

In [7]:
# 隣国の名入れ

neighbors = {
    "TÜRKIYE": [37.5, 37.5], 
    "IRAQ": [34.5, 42.0],         
    "JORDAN": [31.9, 36.5],       
    "LEBANON": [34.2, 35.0]       
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="font-size: 14pt; font-weight: bold; color: gray; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'''
        )
    ).add_to(m)

In [8]:
# シリア14地域の名入れ

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'<div style="font-size: 10px; color: black; font-weight: bold; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'
        )
    ).add_to(m)

In [9]:
# ETL Load

m.save('06_syria_windowedreading.html')